# Semantic Search & LLM Application
**Prototype Asisten FAQ Customer Service E-Commerce**

Notebook ini membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

| Tahap | Implementasi |
|---|---|
| Ingest | Pandas memuat `data/ecommerce_faq_knowledge_base.csv` dan membentuk satu unit teks per FAQ |
| Embed | Gemini Embeddings API (`gemini-embedding-001`), `task_type` dibedakan untuk dokumen dan query |
| Store | FAISS `IndexFlatIP` atas vektor ter-normalisasi (inner product = cosine similarity), disimpan ke `vector_store/` |
| Retrieve | Fungsi `semantic_search()` → top-k FAQ beserta skor cosine similarity |
| Generate | Prompt terstruktur (Persona, Task, Context, Format) → Gemini API |

> Sebelum menjalankan: isi `GEMINI_API_KEY` di file `.env` pada root project (lihat `.env.example`).

## 1. Import Library & Load Knowledge Base

In [1]:
import os
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
import faiss
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Root project (notebook bisa dijalankan dari folder notebook/ maupun dari root)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'ecommerce_faq_knowledge_base.csv'
VECTOR_DIR = ROOT / 'vector_store'
DOC_DIR = ROOT / 'documentation'

load_dotenv(ROOT / '.env')
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY belum diisi di file .env'

EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'gemini-embedding-001')
LLM_MODEL = os.getenv('LLM_MODEL', 'gemini-flash-latest')
TOP_K = 3

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))
print('Embedding model:', EMBEDDING_MODEL)
print('LLM model      :', LLM_MODEL)

Embedding model: gemini-embedding-001
LLM model      : gemini-flash-latest


In [2]:
df = pd.read_csv(DATA_PATH)  # data asli hanya dibaca, tidak diubah
print('Total entri knowledge base:', len(df))
print('Kolom:', list(df.columns))
print('Missing value per kolom:', df.isna().sum().to_dict())
print('Duplikat pertanyaan:', df['question'].duplicated().sum())
df.head()

Total entri knowledge base: 59
Kolom: ['faq_id', 'category', 'question', 'answer']
Missing value per kolom: {'faq_id': 0, 'category': 0, 'question': 0, 'answer': 0}
Duplikat pertanyaan: 0


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [3]:
# Distribusi kategori dan panjang teks
summary = df.groupby('category').agg(jumlah_faq=('faq_id', 'count')).sort_values('jumlah_faq', ascending=False)
print(summary, '\n')
print('Jumlah kategori:', df['category'].nunique())
print('Rata-rata panjang (kata) -> question: {:.1f}, answer: {:.1f}'.format(
    df['question'].str.split().str.len().mean(), df['answer'].str.split().str.len().mean()))

                  jumlah_faq
category                    
Pemesanan                  9
Pengiriman                 9
Pembayaran                 8
Pengembalian               8
Akun                       7
Produk                     7
Promo                      6
Customer Service           5 

Jumlah kategori: 8
Rata-rata panjang (kata) -> question: 7.6, answer: 17.2


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

Kategori juga ikut disertakan di awal teks. Pertanyaan pelanggan sering menyebut topik (mis. "refund", "voucher"), sehingga label kategori membantu vektor menangkap konteks topik. Jawaban ikut di-embed karena banyak kata kunci penting (mis. "Virtual Account", "resi", "kartu kredit") hanya muncul di jawaban, tidak di pertanyaan. Setiap FAQ cukup pendek (rata-rata < 40 kata), jadi tidak perlu chunking.

In [4]:
df['content'] = (
    'Kategori: ' + df['category'] + '\n'
    + 'Pertanyaan: ' + df['question'] + '\n'
    + 'Jawaban: ' + df['answer']
)
documents = df['content'].tolist()
print('Jumlah unit retrievable:', len(documents))
print('Contoh unit retrievable:\n')
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:

Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

- `task_type='RETRIEVAL_DOCUMENT'` untuk entri FAQ dan `task_type='RETRIEVAL_QUERY'` untuk pertanyaan pengguna. Model embedding Gemini mengoptimalkan vektor sesuai perannya (asymmetric retrieval).
- Request dikirim per batch (bukan satu per satu) agar lebih cepat dan hemat kuota.
- Vektor dinormalisasi L2 sehingga inner product = cosine similarity.

In [5]:
def embed_texts(texts, task_type, batch_size=50, retries=3):
    """Mengubah list teks menjadi matriks embedding float32 yang sudah dinormalisasi L2."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(retries):
            try:
                result = client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type),
                )
                vectors.extend(e.values for e in result.embeddings)
                break
            except Exception as err:
                if attempt == retries - 1:
                    raise
                wait = 2 ** attempt * 5
                print(f'Gagal embed ({err}); coba lagi dalam {wait} detik...')
                time.sleep(wait)
    arr = np.array(vectors, dtype='float32')
    faiss.normalize_L2(arr)
    return arr


def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    return embed_texts([text], task_type=task_type)[0]


embeddings = embed_texts(documents, task_type='RETRIEVAL_DOCUMENT')
print('Shape embeddings:', embeddings.shape)
print('Tipe data       :', embeddings.dtype)
print('Norma L2 (min/max):', np.linalg.norm(embeddings, axis=1).min().round(4), '/', np.linalg.norm(embeddings, axis=1).max().round(4))
assert embeddings.shape[0] == len(df), 'Jumlah embedding tidak sama dengan jumlah entri'

Shape embeddings: (59, 3072)
Tipe data       : float32
Norma L2 (min/max): 1.0 / 1.0


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

`IndexFlatIP` melakukan exact search dengan inner product. Karena vektor sudah dinormalisasi, skornya adalah cosine similarity (rentang -1 s.d. 1, makin tinggi makin mirip). Ini lebih mudah diinterpretasi daripada jarak L2. Untuk 59 entri, exact search sudah sangat cepat sehingga index aproksimasi (IVF/HNSW) belum diperlukan.

Index dan metadata disimpan ke folder `vector_store/` agar bisa dipakai ulang tanpa embed ulang.

In [6]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Dimensi vektor              :', dimension)
print('Jumlah vektor tersimpan     :', index.ntotal)
print('Jumlah entri knowledge base :', len(df))
assert index.ntotal == len(df), 'Jumlah vektor di index tidak sesuai jumlah entri knowledge base'

Dimensi vektor              : 3072
Jumlah vektor tersimpan     : 59
Jumlah entri knowledge base : 59


In [7]:
# Simpan index + metadata (mapping posisi vektor -> entri FAQ)
VECTOR_DIR.mkdir(exist_ok=True)
faiss.write_index(index, str(VECTOR_DIR / 'faq.index'))
metadata = {
    'embedding_model': EMBEDDING_MODEL,
    'dimension': dimension,
    'entries': df[['faq_id', 'category', 'question', 'answer']].to_dict(orient='records'),
}
(VECTOR_DIR / 'faq_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

# Verifikasi: muat ulang index dari disk
reloaded = faiss.read_index(str(VECTOR_DIR / 'faq.index'))
print('Index tersimpan di vector_store/faq.index | ntotal setelah reload:', reloaded.ntotal)

Index tersimpan di vector_store/faq.index | ntotal setelah reload: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [8]:
def semantic_search(query, top_k=TOP_K):
    query_embedding = embed_texts([query], task_type='RETRIEVAL_QUERY')
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        row = df.iloc[int(idx)]
        results.append({
            'rank': rank,
            'faq_id': int(row['faq_id']),
            'category': row['category'],
            'question': row['question'],
            'answer': row['answer'],
            'score': round(float(score), 4),  # cosine similarity
        })
    return results


def show_results(results):
    return pd.DataFrame(results)[['rank', 'score', 'faq_id', 'category', 'question']]


# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
show_results(test_results)

,rank,score,faq_id,category,question
0,1,0.7971,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...
1,2,0.7816,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...
2,3,0.7593,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

| Bagian | Isi | Tujuan |
|---|---|---|
| **Persona** | Asisten CS e-commerce yang ramah, berbahasa Indonesia | Menjaga nada dan bahasa jawaban |
| **Task** | Jawab *hanya* dari Context, pilih FAQ yang relevan, akui jika tidak tahu, jangan mengarang angka/kebijakan | Grounding: mencegah halusinasi dan jawaban dari pengetahuan umum LLM |
| **Context** | Top-k FAQ hasil retrieval beserta ID, kategori, dan skor kemiripan | Sumber fakta satu-satunya; ID dipakai untuk sitasi |
| **Format** | Maks. 4 kalimat, daftar bernomor untuk langkah, baris `Sumber: FAQ #id` | Jawaban ringkas dan dapat ditelusuri ke FAQ asal |

Temperature diset rendah (0.2) agar jawaban konsisten dan tidak "kreatif" di luar Context.

In [9]:
PROMPT_TEMPLATE = '''### PERSONA
Kamu adalah "Asisten CS", customer service virtual sebuah platform e-commerce di Indonesia.
Kamu ramah, sopan, empatik, dan menjawab dalam Bahasa Indonesia yang mudah dipahami pelanggan.

### TASK
Jawab pertanyaan pelanggan di bawah ini dengan aturan berikut:
1. Gunakan HANYA informasi yang ada di bagian CONTEXT. Jangan memakai pengetahuan umum atau asumsi di luar CONTEXT.
2. CONTEXT berisi beberapa FAQ hasil pencarian; tidak semuanya pasti relevan. Pakai hanya FAQ yang benar-benar menjawab pertanyaan.
3. Jika CONTEXT tidak memuat jawabannya, katakan dengan jujur: "Maaf, informasi tersebut belum tersedia di FAQ kami." lalu sarankan pelanggan menghubungi customer service.
4. Jika CONTEXT hanya menjawab sebagian, jawab bagian yang tersedia dan sebutkan bagian mana yang belum ada informasinya.
5. Jangan mengarang angka, biaya, durasi, nama fitur, atau kebijakan yang tidak tertulis di CONTEXT.

### CONTEXT
{context}

### PERTANYAAN PELANGGAN
{query}

### FORMAT JAWABAN
- Maksimal 4 kalimat, natural dan langsung ke inti.
- Jika jawabannya berupa langkah-langkah, gunakan daftar bernomor.
- Jangan menyebut kata "CONTEXT" atau "dokumen" kepada pelanggan.
- Akhiri dengan satu baris: "Sumber: FAQ #<id>" (pisahkan dengan koma bila lebih dari satu), atau "Sumber: -" jika tidak ada FAQ yang relevan.
'''


def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ #{r['faq_id']} | Kategori: {r['category']} | Skor kemiripan: {r['score']:.3f}]\n"
        f"Pertanyaan: {r['question']}\nJawaban: {r['answer']}"
        for r in search_results
    )
    return PROMPT_TEMPLATE.format(context=context, query=query)


print(build_prompt('kenapa paket saya belum sampai?', test_results))

### PERSONA
Kamu adalah "Asisten CS", customer service virtual sebuah platform e-commerce di Indonesia.
Kamu ramah, sopan, empatik, dan menjawab dalam Bahasa Indonesia yang mudah dipahami pelanggan.

### TASK
Jawab pertanyaan pelanggan di bawah ini dengan aturan berikut:
1. Gunakan HANYA informasi yang ada di bagian CONTEXT. Jangan memakai pengetahuan umum atau asumsi di luar CONTEXT.
2. CONTEXT berisi beberapa FAQ hasil pencarian; tidak semuanya pasti relevan. Pakai hanya FAQ yang benar-benar menjawab pertanyaan.
3. Jika CONTEXT tidak memuat jawabannya, katakan dengan jujur: "Maaf, informasi tersebut belum tersedia di FAQ kami." lalu sarankan pelanggan menghubungi customer service.
4. Jika CONTEXT hanya menjawab sebagian, jawab bagian yang tersedia dan sebutkan bagian mana yang belum ada informasinya.
5. Jangan mengarang angka, biaya, durasi, nama fitur, atau kebijakan yang tidak tertulis di CONTEXT.

### CONTEXT
[FAQ #12 | Kategori: Pengiriman | Skor kemiripan: 0.797]
Pertanyaan: Bag

## 7. Mengintegrasikan LLM

In [10]:
def call_llm(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.2),
            )
            return (response.text or '').strip()
        except Exception as err:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt * 5
            print(f'Gagal memanggil LLM ({err}); coba lagi dalam {wait} detik...')
            time.sleep(wait)


def generate_answer(query, top_k=TOP_K):
    """Alur lengkap: query -> retrieval -> prompt -> jawaban LLM."""
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)
    answer = call_llm(prompt)
    return answer, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print(f"- FAQ #{s['faq_id']} (score {s['score']:.3f}): {s['question']}")

Gagal memanggil LLM (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); coba lagi dalam 5 detik...


Jawaban LLM:
Halo! Ya, Anda bisa membayar menggunakan cicilan. Fitur cicilan ini tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank.

Sumber: FAQ #20

Sumber FAQ yang digunakan:
- FAQ #20 (score 0.807): Apakah saya bisa membayar dengan cicilan?
- FAQ #17 (score 0.694): Metode pembayaran apa saja yang tersedia?
- FAQ #22 (score 0.688): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

Delapan query uji dibagi ke tiga kelompok. `expected` berisi ID FAQ yang seharusnya ter-retrieve (ditentukan manual dengan membaca knowledge base). Kosong berarti jawabannya tidak ada di knowledge base.

| # | Tipe | Tujuan |
|---|---|---|
| Q1–Q2 | Langsung | Kalimat mirip FAQ asli. Baseline: harus tepat di peringkat 1 |
| Q3–Q6 | Parafrase / sinonim / bahasa informal | Menguji pemahaman makna: kata kuncinya hampir tidak ada yang sama dengan FAQ |
| Q7 | Di luar KB (menjebak) | Topik mirip FAQ yang ada (metode pembayaran) tetapi detailnya tidak ada |
| Q8 | Di luar KB | Topik yang sama sekali tidak dibahas |

In [11]:
test_queries = [
    {'id': 'Q1', 'type': 'Langsung',  'query': 'Metode pembayaran apa saja yang tersedia?', 'expected': [17]},
    {'id': 'Q2', 'type': 'Langsung',  'query': 'Bagaimana cara membatalkan pesanan?', 'expected': [3]},
    {'id': 'Q3', 'type': 'Parafrase', 'query': 'Saldo saya sudah kepotong tapi orderan masih tertulis belum dibayar, gimana nih?', 'expected': [21]},
    {'id': 'Q4', 'type': 'Parafrase', 'query': 'Barang yang datang beda sama yang saya beli', 'expected': [54, 24]},
    {'id': 'Q5', 'type': 'Parafrase', 'query': 'Bisa nyicil nggak kalau belanja di sini?', 'expected': [20]},
    {'id': 'Q6', 'type': 'Parafrase', 'query': 'Saya kelupaan kata sandi jadi tidak bisa masuk ke akun', 'expected': [32, 55]},
    {'id': 'Q7', 'type': 'Di luar KB', 'query': 'Apakah bisa bayar pakai Bitcoin atau crypto lainnya?', 'expected': []},
    {'id': 'Q8', 'type': 'Di luar KB', 'query': 'Jam berapa toko offline kalian buka di Jakarta?', 'expected': []},
]

test_log = []
for t in test_queries:
    answer, sources = generate_answer(t['query'])
    retrieved = [s['faq_id'] for s in sources]
    test_log.append({**t, 'answer': answer, 'sources': sources})

    print(f"{t['id']} [{t['type']}] {t['query']}")
    for s in sources:
        mark = '✓' if s['faq_id'] in t['expected'] else ' '
        print(f"   {mark} #{s['rank']} score={s['score']:.3f}  FAQ {s['faq_id']:>2} ({s['category']}): {s['question']}")
    print('Jawaban:', answer)
    print('-' * 100)
    time.sleep(1)  # jeda singkat agar tidak terkena rate limit

Gagal memanggil LLM (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); coba lagi dalam 5 detik...


Gagal memanggil LLM (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); coba lagi dalam 10 detik...


Q1 [Langsung] Metode pembayaran apa saja yang tersedia?
   ✓ #1 score=0.796  FAQ 17 (Pembayaran): Metode pembayaran apa saja yang tersedia?
     #2 score=0.726  FAQ 22 (Pembayaran): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
     #3 score=0.706  FAQ 20 (Pembayaran): Apakah saya bisa membayar dengan cicilan?
Jawaban: Halo! Saat ini kami menyediakan metode pembayaran melalui transfer bank, kartu kredit/debit, e-wallet, serta pembayaran melalui minimarket rekanan. Silakan pilih metode yang paling nyaman untuk transaksi Anda, ya.

Sumber: FAQ #17
----------------------------------------------------------------------------------------------------


Q2 [Langsung] Bagaimana cara membatalkan pesanan?
   ✓ #1 score=0.780  FAQ  3 (Pemesanan): Bagaimana cara membatalkan pesanan yang sudah dibuat?
     #2 score=0.714  FAQ 24 (Pengembalian): Bagaimana prosedur pengembalian barang yang tidak sesuai?
     #3 score=0.703  FAQ  8 (Pemesanan): Kenapa pesanan saya otomatis dibatalkan sistem?
Jawaban: Halo! Kamu dapat membatalkan pesanan melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Mohon diperhatikan bahwa setelah status berubah menjadi 'Dikirim', pembatalan sudah tidak dapat dilakukan lagi.

Sumber: FAQ #3
----------------------------------------------------------------------------------------------------


Q3 [Parafrase] Saldo saya sudah kepotong tapi orderan masih tertulis belum dibayar, gimana nih?
   ✓ #1 score=0.804  FAQ 21 (Pembayaran): Kenapa pembayaran saya berhasil dipotong tapi status pesanan masih belum terbayar?
     #2 score=0.709  FAQ  8 (Pemesanan): Kenapa pesanan saya otomatis dibatalkan sistem?
     #3 score=0.706  FAQ 19 (Pembayaran): Berapa lama batas waktu pembayaran sebelum pesanan dibatalkan?
Jawaban: Halo, mohon maaf atas ketidaknyamanan yang dialami. Hal ini biasanya terjadi karena adanya keterlambatan sinkronisasi pada sistem pembayaran. Silakan tunggu 15-30 menit atau hubungi customer service jika status pesanan masih belum berubah.

Sumber: FAQ #21
----------------------------------------------------------------------------------------------------


Q4 [Parafrase] Barang yang datang beda sama yang saya beli
   ✓ #1 score=0.799  FAQ 54 (Pengembalian): Saya menerima barang yang salah, apa yang harus saya lakukan?
   ✓ #2 score=0.737  FAQ 24 (Pengembalian): Bagaimana prosedur pengembalian barang yang tidak sesuai?
     #3 score=0.715  FAQ 30 (Pengembalian): Apakah bisa menukar barang dengan ukuran/warna lain daripada refund?
Jawaban: Mohon maaf atas ketidaknyamanan yang Anda alami, silakan ikuti langkah-langkah berikut:
1. Buka menu 'Pesanan Saya' dan ajukan komplain 'Barang Tidak Sesuai'.
2. Sertakan bukti foto barang yang diterima untuk mempercepat proses verifikasi.
Pastikan pengajuan pengembalian ini dilakukan maksimal 3 hari sejak barang diterima.

Sumber: FAQ #24, FAQ #54
----------------------------------------------------------------------------------------------------


Q5 [Parafrase] Bisa nyicil nggak kalau belanja di sini?
   ✓ #1 score=0.797  FAQ 20 (Pembayaran): Apakah saya bisa membayar dengan cicilan?
     #2 score=0.700  FAQ 22 (Pembayaran): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
     #3 score=0.695  FAQ 17 (Pembayaran): Metode pembayaran apa saja yang tersedia?
Jawaban: Halo Kak, bisa ya! Fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu dengan nominal minimal yang ditentukan oleh pihak bank. 

Sumber: FAQ #20
----------------------------------------------------------------------------------------------------


Q6 [Parafrase] Saya kelupaan kata sandi jadi tidak bisa masuk ke akun
   ✓ #1 score=0.778  FAQ 32 (Akun): Saya lupa password akun, bagaimana cara mengaturnya ulang?
   ✓ #2 score=0.748  FAQ 55 (Akun): Kenapa akun saya tiba-tiba tidak bisa login?
     #3 score=0.676  FAQ 36 (Akun): Apakah data pribadi saya aman di platform ini?
Jawaban: Halo, Kak. Jangan khawatir, Anda dapat mengatur ulang kata sandi akun Anda dengan langkah-langkah berikut:

1. Klik 'Lupa Password' pada halaman login.
2. Masukkan email terdaftar.
3. Ikuti instruksi reset password yang dikirim ke email Anda.

Sumber: FAQ #32
----------------------------------------------------------------------------------------------------


Q7 [Di luar KB] Apakah bisa bayar pakai Bitcoin atau crypto lainnya?
     #1 score=0.706  FAQ 17 (Pembayaran): Metode pembayaran apa saja yang tersedia?
     #2 score=0.677  FAQ 22 (Pembayaran): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
     #3 score=0.673  FAQ 18 (Pembayaran): Apakah ada biaya tambahan untuk pembayaran menggunakan kartu kredit?
Jawaban: Halo, terima kasih telah menghubungi kami. Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi customer service kami untuk mendapatkan bantuan dan informasi lebih lanjut mengenai metode pembayaran tersebut.

Sumber: -
----------------------------------------------------------------------------------------------------


Q8 [Di luar KB] Jam berapa toko offline kalian buka di Jakarta?
     #1 score=0.635  FAQ 43 (Produk): Apakah stok yang tertera di aplikasi selalu update secara real-time?
     #2 score=0.632  FAQ  9 (Pengiriman): Berapa lama estimasi waktu pengiriman barang?
     #3 score=0.629  FAQ 58 (Pemesanan): Apakah bisa memesan dalam jumlah besar untuk kebutuhan reseller?
Jawaban: Halo! Maaf, informasi tersebut belum tersedia di FAQ kami. Silakan hubungi tim customer service kami untuk bantuan dan informasi lebih lanjut.

Sumber: -
----------------------------------------------------------------------------------------------------


In [12]:
# Ringkasan metrik retrieval
rows = []
for t in test_log:
    ids = [s['faq_id'] for s in t['sources']]
    has_answer = bool(t['expected'])
    rows.append({
        'id': t['id'],
        'type': t['type'],
        'expected': t['expected'] or '-',
        'retrieved_top3': ids,
        'top1_score': t['sources'][0]['score'],
        'hit@1': (ids[0] in t['expected']) if has_answer else None,
        'hit@3': any(i in t['expected'] for i in ids) if has_answer else None,
        'llm_says_unavailable': 'belum tersedia' in t['answer'].lower(),
    })
summary_df = pd.DataFrame(rows)

answerable = summary_df[summary_df['type'] != 'Di luar KB']
print('Hit@1 (query yang ada jawabannya):', f"{answerable['hit@1'].mean():.0%}")
print('Hit@3 (query yang ada jawabannya):', f"{answerable['hit@3'].mean():.0%}")
print('Rata-rata top-1 score  | ada jawaban:', round(answerable['top1_score'].mean(), 3),
      '| di luar KB:', round(summary_df[summary_df['type'] == 'Di luar KB']['top1_score'].mean(), 3))
summary_df

Hit@1 (query yang ada jawabannya): 100%
Hit@3 (query yang ada jawabannya): 100%
Rata-rata top-1 score  | ada jawaban: 0.792 | di luar KB: 0.671


,id,type,expected,retrieved_top3,top1_score,hit@1,hit@3,llm_says_unavailable
0,Q1,Langsung,[17],"[17, 22, 20]",0.7957,True,True,False
1,Q2,Langsung,[3],"[3, 24, 8]",0.7796,True,True,False
2,Q3,Parafrase,[21],"[21, 8, 19]",0.8038,True,True,False
3,Q4,Parafrase,"[54, 24]","[54, 24, 30]",0.7992,True,True,False
4,Q5,Parafrase,[20],"[20, 22, 17]",0.7974,True,True,False
5,Q6,Parafrase,"[32, 55]","[32, 55, 36]",0.7784,True,True,False
6,Q7,Di luar KB,-,"[17, 22, 18]",0.7062,None,None,True
7,Q8,Di luar KB,-,"[43, 9, 58]",0.6351,None,None,True


In [13]:
# Simpan hasil mentah pengujian untuk dokumentasi
DOC_DIR.mkdir(exist_ok=True)
(DOC_DIR / 'test_results_raw.json').write_text(
    json.dumps({'embedding_model': EMBEDDING_MODEL, 'llm_model': LLM_MODEL, 'results': test_log},
               ensure_ascii=False, indent=2, default=str),
    encoding='utf-8')
print('Hasil pengujian disimpan ke documentation/test_results_raw.json')

Hasil pengujian disimpan ke documentation/test_results_raw.json


### Catatan Analisis Relevansi

#### Ringkasan Hasil

| ID | Tipe | Query | Top-3 FAQ (skor cosine) | Hit@1 | Respons LLM |
|---|---|---|---|---|---|
| Q1 | Langsung | Metode pembayaran apa saja yang tersedia? | **17** (0.796), 22 (0.726), 20 (0.706) | ✅ | Tepat, menyebut 4 metode dari FAQ #17 |
| Q2 | Langsung | Bagaimana cara membatalkan pesanan? | **3** (0.780), 24 (0.714), 8 (0.703) | ✅ | Tepat, termasuk syarat status 'Diproses' |
| Q3 | Parafrase | Saldo saya sudah kepotong tapi orderan masih tertulis belum dibayar, gimana nih? | **21** (0.804), 8 (0.709), 19 (0.706) | ✅ | Tepat, "tunggu 15–30 menit" sesuai FAQ |
| Q4 | Parafrase | Barang yang datang beda sama yang saya beli | **54** (0.799), **24** (0.737), 30 (0.715) | ✅ | Menggabungkan FAQ #54 + #24 menjadi langkah bernomor |
| Q5 | Parafrase | Bisa nyicil nggak kalau belanja di sini? | **20** (0.797), 22 (0.700), 17 (0.695) | ✅ | Tepat, syarat kartu kredit dan nominal minimal |
| Q6 | Parafrase | Saya kelupaan kata sandi jadi tidak bisa masuk ke akun | **32** (0.778), **55** (0.748), 36 (0.676) | ✅ | Tepat, 3 langkah reset password |
| Q7 | Di luar KB | Apakah bisa bayar pakai Bitcoin atau crypto lainnya? | 17 (0.706), 22 (0.677), 18 (0.673) | – | Menolak dengan jujur, mengarahkan ke CS |
| Q8 | Di luar KB | Jam berapa toko offline kalian buka di Jakarta? | 43 (0.635), 9 (0.632), 58 (0.629) | – | Menolak dengan jujur, mengarahkan ke CS |

**Metrik retrieval (6 query yang ada jawabannya):** Hit@1 = **100%**, Hit@3 = **100%**. Rata-rata skor top-1: **0.792** untuk query yang ada jawabannya, **0.671** untuk query di luar KB.

#### Analisis

**1. Semantic search menangkap makna, bukan sekadar kata kunci.**
Keempat query parafrase memakai bahasa informal dan sinonim yang hampir tidak berbagi kata dengan FAQ, tetapi semuanya tetap tepat di peringkat 1:
- Q3: *"saldo kepotong"*, *"orderan"* → FAQ #21 *"pembayaran berhasil dipotong"*, *"status pesanan"*. Skor top-1 justru yang tertinggi dari semua query (0.804).
- Q5: slang *"nyicil"* → FAQ #20 *"cicilan"* (0.797).
- Q6: *"kata sandi"*, *"masuk"* → FAQ #32 *"password"*, *"login"*. Ini padanan lintas bahasa (Indonesia–Inggris) yang tidak mungkin tertangkap pencarian kata kunci biasa.
- Q4 dan Q6 masing-masing punya dua FAQ relevan, dan **keduanya masuk top-2**.

**2. Pemisahan peringkat cukup jelas.** Selisih skor top-1 dengan top-2 adalah 0.06–0.10 (Q1 0.070, Q2 0.066, Q3 0.095, Q5 0.097). Pengecualiannya Q6 (0.030), tetapi di sana FAQ #2 (#55, akun tidak bisa login) memang juga relevan.

**3. Masih ada noise di peringkat 2–3.** Pada Q2 ("membatalkan pesanan"), peringkat 2 justru FAQ #24 (*prosedur pengembalian barang*, kategori lain), di atas FAQ #8 (*pesanan otomatis dibatalkan*) yang lebih dekat topiknya. Embedding menganggap "membatalkan" dan "mengembalikan" berdekatan secara semantik. Noise ini tidak merusak jawaban karena aturan Task #2 ("pakai hanya FAQ yang benar-benar menjawab") membuat LLM mengabaikan FAQ tersebut. Dengan kata lain, LLM ikut berperan sebagai filter relevansi.

**4. Query di luar KB: retrieval tetap mengembalikan hasil, LLM yang menolak.**
FAISS selalu mengembalikan *k* hasil terdekat dan tidak punya konsep "tidak ada jawaban":
- Q7 (crypto) mengambil FAQ seputar pembayaran dengan skor 0.706. Topiknya dekat, tetapi tidak menjawab. Ini kasus paling menjebak: dari sisi retrieval terlihat "cukup relevan".
- Q8 (jam toko offline) mengambil FAQ yang tidak berhubungan (stok, estimasi kirim, reseller) dengan skor terendah 0.635.

Pada kedua kasus LLM **tidak berhalusinasi**. LLM tidak mengarang "kami tidak menerima crypto" atau jam buka, melainkan menjawab dengan kalimat penolakan sesuai prompt dan mencantumkan `Sumber: -`. Namun pada Q7 jawabannya **terlalu konservatif**. Sebenarnya LLM bisa menyebutkan metode yang tersedia (FAQ #17) dan menjelaskan bahwa crypto tidak tercantum, sesuai aturan Task #4 (jawaban parsial), tetapi aturan itu tidak diterapkan.

**5. Skor cosine terkompresi, jadi nilai absolutnya harus dibaca relatif.** Teks yang tidak berhubungan sama sekali (Q8) masih mendapat skor ±0.63, sementara FAQ yang tepat ±0.78–0.80. Pada sampel ini, ambang ±0.75 sudah memisahkan query yang ada jawabannya (min 0.778) dari yang di luar KB (maks 0.706). Tetapi 8 query terlalu sedikit untuk menetapkan ambang. Q7 menunjukkan query di luar KB yang topiknya bertetangga bisa mendekati skor query yang relevan.

**6. Kualitas jawaban LLM: grounded dan bisa ditelusuri.** Setiap fakta di jawaban sudah dicocokkan dengan teks FAQ asli: "15–30 menit" (Q3 → FAQ #21), "maksimal 3 hari sejak barang diterima" (Q4 → FAQ #24), langkah reset password (Q6 → FAQ #32). Tidak ada angka atau kebijakan yang dikarang. Kedelapan jawaban mencantumkan sumber, dan format daftar bernomor dipakai saat jawabannya berupa langkah (Q4, Q6). Pada Q4, LLM menggabungkan dua FAQ (komplain "Barang Tidak Sesuai" dari #54 dan batas 3 hari dari #24) lalu mengutip keduanya. Ini sintesis yang membantu, tetapi berisiko kecil mencampur dua kebijakan yang sebenarnya berbeda (komplain vs pengembalian).

**7. Kelemahan gaya:** sapaan tidak konsisten antar jawaban ("Kak", "Kamu", "Anda", bahkan campur dalam satu jawaban di Q6), karena Persona tidak menetapkan kata ganti.

**8. Catatan operasional:** selama pengujian, API Gemini beberapa kali mengembalikan `503 UNAVAILABLE` (high demand). Mekanisme retry dengan backoff di `call_llm()` berhasil menanganinya sehingga notebook tetap selesai tanpa error.


#### Keterbatasan Sistem

1. **Tidak ada ambang relevansi.** FAISS selalu mengembalikan top-k meskipun tidak ada yang relevan (Q7, Q8), sehingga penolakan sepenuhnya bergantung pada kepatuhan LLM terhadap prompt. Untuk produksi perlu ambang skor yang dikalibrasi sebagai lapisan pengaman kedua.
2. **Skor cosine terkompresi** (±0.63 untuk teks tak berhubungan, ±0.80 untuk yang tepat). Skor tidak bisa dibaca sebagai probabilitas dan perlu kalibrasi dengan set uji yang lebih besar.
3. **Evaluasi masih kecil:** 8 query dengan label relevansi manual. Deteksi penolakan (`llm_says_unavailable`) memakai pencocokan string, sehingga belum menilai kualitas jawaban secara otomatis.
4. **Noise antar kategori** di peringkat bawah (Q2: FAQ pengembalian muncul untuk query pembatalan). Belum ada re-ranking atau hybrid search (BM25 + vektor) untuk mempertajam urutan.
5. **LLM terlalu konservatif untuk jawaban parsial** (Q7): langsung menolak padahal sebagian informasi tersedia. Sapaan juga tidak konsisten.
6. **Penggabungan beberapa FAQ** bisa mencampur kebijakan yang berbeda (Q4).
7. **Knowledge base kecil dan statis** (59 entri, jawaban umum seperti "biasanya"). Tidak terhubung ke data transaksi, sehingga tidak bisa menjawab kasus spesifik seperti status pesanan tertentu. Setiap perubahan FAQ memerlukan embedding ulang.
8. **Single-turn**, tanpa memori percakapan, sehingga pertanyaan lanjutan ("kalau yang tadi gimana?") tidak bisa dipahami.
9. **Ketergantungan pada API eksternal:** ada error 503 dan rate limit. Alias `gemini-flash-latest` bisa berpindah versi sehingga jawaban bisa berubah antar run. Untuk hasil yang reproducible, versi model sebaiknya di-pin.

**Arah pengembangan (menuju RAG penuh):** ambang skor + fallback ke CS, hybrid search, re-ranker, penetapan sapaan di Persona, contoh jawaban parsial (few-shot), evaluasi lebih besar (mis. LLM-as-judge untuk faithfulness), dan pin versi model.
